In [1]:
import os
import langchain
from langchain_community.document_loaders import PyPDFLoader,PyPDFium2Loader,PyMuPDFLoader,TextLoader,CSVLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders.excel import UnstructuredExcelLoader
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from typing import List,Any

def load_all_documents(data_dir:str)->List[Any]:
    """load all documents  fromk the data directory and convert to Lanchain document strcuture
    Supported :PDF,JSON,CSV,EXCEL ,Word """

    ## use  project root folder to access data

    data_path=Path(data_dir).resolve()
    print(f"[DEBUG]Data Path:{data_path}")
    documents=[]

    #find all PDF files in loop
    pdf_files=list(data_path.glob("**/*.pdf"))
    print(f"[DEBUG] found :{len(pdf_files)} PDF Files : {[str(f) for f in pdf_files]}")
    for pdf in pdf_files:
        print (f"processing of : {pdf.name}")
        try:
            loader=PyPDFLoader(str(pdf))
            loaded=loader.load()
            print(f"[DEBUG] Loaded :{len(loaded)} PDF doc from {pdf}")
            documents.extend(loaded) ## appending the documents
        except Exception as e:
            print (f"[Error] FAILED TO LOAD PDF FILE {pdf}:{e}")

    ### json files
    json_files=list(data_path.glob("**/*.json"))
    print(f"[DEBUG] found :{len(json_files)} JSON Files : {[str(f) for f in json_files]}")
    for file_path in json_files:
        print (f"processing of : {file_path.name}")
        try:
            loader=JSONLoader(str(file_path),
                                jq_schema=".[]",
                                text_content=False
                              )
            loaded=loader.load()
            print(f"[DEBUG] Loaded :{len(loaded)} jSON doc from {file_path}")
            documents.extend(loaded) ## appending the documents
        except Exception as e:
            print (f"[Error] FAILED TO LOAD JSON FILE {file_path}:{e}")
    
    return documents

d:\farzana\vr_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
docs=load_all_documents("data")
docs

[DEBUG]Data Path:D:\farzana\vr_rag\testfiles\data
[DEBUG] found :0 PDF Files : []
[DEBUG] found :0 JSON Files : []


[]

In [3]:
import numpy as np
import langchain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List,Any   
from sentence_transformers import SentenceTransformer # embedding model will be available inside this
import uuid #everyrecord inserted to db will have an id will use this library
from uuid import uuid4
#from .data_loader import load_all_documents



class EmbeddingPipeline:
    """Handles document embedding generation using sentence transformer"""

    def __init__(
        self,
        model_name: str = "all-MiniLM-L6-v2",
        chunk_size: int = 1000,
        chunk_overlap: int = 200
    ):
        """Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.model = SentenceTransformer(model_name)
        print(f"Loaded model: {model_name}")

    def chunk_documents(self,documents:list[Any])->list[Any]:
        """Chunk documents into smaller pieces for embedding generation
        Args:
            documents:List of langchain document objects"""
        splitter=RecursiveCharacterTextSplitter(chunk_size=self.chunk_size,
                                                chunk_overlap=self.chunk_overlap,
                                                separators=["\n\n","\n",".","!","?"," ", ""])
        chunks=splitter.split_documents(documents)
        print(f"Chunked {len(documents)} documents into {len(chunks)} chunks")
        return chunks
   
   
    def embed_documents(self,chunks:list[Any])->np.ndarray:
        """Gnerate embedding for the chunks of documents aRGS: dOCUMENT CHUNKS WHICH ARE split by 
        langchain text splitter"""
        texts=[chunk.page_content for chunk in chunks]
        print(f"Info:Generating embeddings for {len(texts)} chunks")
        embeddings = self.model.encode([chunk.page_content for chunk in chunks], show_progress_bar=True)
        print(f"Info:Created embeddings for {len(embeddings)} chunks")
        return embeddings

     
     

In [4]:
embedding=EmbeddingPipeline()
embedding

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11403.28it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: all-MiniLM-L6-v2


In [7]:
import os
import faiss ## Facebook AI Similarity Search
import numpy as np
import pickle ## serialize the vector store to disk
import chromadb ## open source vector database 
from typing import List,Any
from sentence_transformers import SentenceTransformer
#from embedding import EmbeddingPipeline

### all-MiniLM-L6-v2 is a popular sentence transformer model that generates 384-dimensional dense vector embeddings for sentences and paragraphs.
#  It is designed to capture semantic meaning and relationships between text, making it useful for tasks like information retrieval, clustering, and natural language understanding.
class VectorStore:
    """Handles document embedding generation using sentence transformer"""
    def __init__(self,persist_directory:str ="faiss_store",embedding_model:str="all-MiniLM-L6-v2",chunk_size:int=1000,chunk_overlap:int=200):
     """Initialize the vector store
     """
     self.persist_directory=persist_directory
     os.makedirs(self.persist_directory,exist_ok=True)
     self.metadata=[]
     self.embedding_model=embedding_model
     self.model=SentenceTransformer(embedding_model)
     self.chunk_size=chunk_size
     self.chunk_overlap=chunk_overlap
     self.index = None
     print(f"Info:Loaded embedding model : {embedding_model}")

    def build_from_documents(self,documents:List[Any]):
       print(f"Info:Building vector store from: {len(documents)} raw documents")
       emb_pipeline=EmbeddingPipeline(model_name=self.embedding_model,chunk_size=self.chunk_size,chunk_overlap=self.chunk_overlap)
       chunks=emb_pipeline.chunk_documents(documents)
       embeddings=emb_pipeline.embed_documents(chunks)
       metadatas=[{"text":chunk.page_content} for chunk in chunks]
       self.add_embeddings(np.array(embeddings).astype('float32'),metadatas)
       self.save()
       print(f"Info: vector store build and saved to: {self.persist_directory} ")

    def add_embeddings(self,embeddings:np.ndarray,metadatas:List[Any]=None):
        """Add embeddings and their metadata to the vector store"""
        dim=embeddings.shape[1]
        if self.index is None:
           self.index=faiss.IndexFlatL2(dim) ## L2 distance metric
        self.index.add(embeddings)
        if metadatas:
           self.metadata.extend(metadatas)
        print(f"Info:Added {embeddings.shape[0]} embeddings to the vector store. Total embeddings")

       #### pickle module converts Python objects to a byte stream and restores them later, allowing to save and load complex data structures like lists,
       #  dictionaries, and custom objects with ease.
    def save(self):
        """Save the vector store to disk"""
        faiss_path=os.path.join(self.persist_directory,"faiss.index") ## vectorstore
        meta_path=os.path.join(self.persist_directory,"metadata.pkl") ## metadata for the documents
        faiss.write_index(self.index,faiss_path)
        with open(meta_path,"wb") as f:
           pickle.dump(self.metadata,f)
        print(f"Info:Saved Faiss Index successfully to {self.persist_directory}")

    def load(self):
       faiss_path=os.path.join(self.persist_directory,"faiss.index")
       meta_path=os.path.join(self.persist_directory,"metadata.pkl") 
       ###INDEX PATH
       self.index = faiss.read_index(faiss_path)
       ###METADATA LOAD
       with open(meta_path,"rb") as f:
            self.metadata=pickle.load(f)
       print(f"Info:Loaded metadata with {len(self.metadata)} entries from {self.persist_directory}")

    def search(self,query_embedding:np.array,top_k:int=5):
       if self.index is None:
        raise ValueError("FAISS index is not initialized. Call build_from_documents() or load() first.")
       D,I =self.index.search(query_embedding,top_k)
       results=[]
       for idx ,dist in zip(I[0],D[0]):
          meta=self.metadata[idx] if idx < len(self.metadata) else None
          results.append({"index":idx,"distance":dist,"metadata":meta})
       return results
          
    def query(self,query_text:str,top_k:int=5):
       print("DEBUG index:", self.index)
       if self.index is None:
        raise ValueError("FAISS index is not initialized. Call build_from_documents() or load() first.")
       print(f"Info:Querying vector store with: {query_text}")
       query_embedding=self.model.encode([query_text],show_progress_bar=False).astype('float32')
       return self.search(query_embedding,top_k)
     
        


In [8]:
vector=VectorStore()
vector

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12773.90it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Info:Loaded embedding model : all-MiniLM-L6-v2


In [9]:
from langchain_groq import ChatGroq ### provides a Language Processing Unit (LPU) inference engine that offers incredibly high-speed inference for large language models
import os
from dotenv import load_dotenv ##  file is specifically used to store all the variables like API keys 
#from src.vector_store import VectorStore
#from src.data_loader import load_all_documents



load_dotenv()

class RagSearch:
    def __init__(self, persist_dir: str = "faiss_store", embedding_model: str = "all-MiniLM-L6-v2", llm_model: str = "llama-3.1-8b-instant"):
        self.vector_store=VectorStore(persist_directory=persist_dir, embedding_model=embedding_model)
##load or build store
        faiss_path = os.path.join(persist_dir, "faiss.index")
        meta_path = os.path.join(persist_dir, "metadata.pkl")
        if os.path.exists(faiss_path) and os.path.exists(meta_path):
          print("[INFO] Loading existing FAISS store...")
          self.vector_store.load()
        else:
         print("[INFO] Building FAISS store...")
         docs = load_all_documents("data")
         self.vector_store.build_from_documents(docs)
        groq_api_key = ""
        self.llm = ChatGroq(groq_api_key=groq_api_key, model_name=llm_model)
        print(f"[INFO] Groq LLM initialized: {llm_model}")


    def search_and_summarize(self, query: str, top_k: int = 5) -> str:
        results = self.vector_store.query(query, top_k=top_k)
        texts = [r["metadata"].get("text", "") for r in results if r["metadata"]]
        context = "\n\n".join(texts)
        if not context:
            return "No relevant documents found."
        prompt = f"""Summarize the following context for the query: '{query}'\n\nContext:\n{context}\n\nSummary:"""
        response = self.llm.invoke([prompt])
        return response.content

# egusage
if __name__ == "__main__":
    rag_search = RagSearch()
    query = "What is situation of patient John Doe?"
    summary = rag_search.search_and_summarize(query, top_k=3)
    print("Summary:", summary)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11892.67it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Info:Loaded embedding model : all-MiniLM-L6-v2
[INFO] Building FAISS store...
[DEBUG]Data Path:D:\farzana\vr_rag\testfiles\data
[DEBUG] found :0 PDF Files : []
[DEBUG] found :0 JSON Files : []
Info:Building vector store from: 0 raw documents


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12067.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: all-MiniLM-L6-v2
Chunked 0 documents into 0 chunks
Info:Generating embeddings for 0 chunks


Batches: 0it [00:00, ?it/s]

Info:Created embeddings for 0 chunks


IndexError: tuple index out of range